In [164]:
function LU_IR(A, b; ul = Float32, u = Float32, ur = Float64, imax=5, atol=1e-6)

    # Step 1: factorization in ul precision
    LU = lu(ul.(A))

    # Step 2: solve x in ul precision
    x = LU \ ul.(b) # should I keep x in higher precision after the operation? 

    for i in 1:imax
        # Step 3: compute residual in ur precision
        r = ur.(b) - ur.(A) * ur.(x)

        # Step 4: solve d in ul precision
        d = LU \ ul.(r)
        
        # Step 5: update x in u precision
        x = u.(x) + u.(d)
        
        #if abs(norm(r)) < atol
            #println("Converged at iter $(i)")
            #break
        #end
    end

    return x
end

LU_IR (generic function with 1 method)

In [166]:
function example(n, m; ul = Float32, u = Float32, ur = Float64, imax=10, atol=1e-6)

    A = rand(n, m)
    b = rand(n)

    start_time = time()
    x_true = A \ b
    elapsed = time() - start_time
    x_true = vec(x_true)
    println("Run time: ", elapsed)

    start_time = time()
    x = LU_IR(A, b)
    elapsed = time() - start_time
    println("Run time: ", elapsed)
    
    error = abs(norm(x-x_true))
    println("Error: ", error)
    

end    

example (generic function with 1 method)

In [180]:
using LinearAlgebra
using BenchmarkTools

example(100, 100)

Run time: 0.0002040863037109375
Run time: 0.0013048648834228516
Error: 4.1409256704819085e-7


In [264]:
A = rand(100, 100)
B = rand(100, 100)

A_16 = Float16.(A)
B_16 = Float16.(B)

A_32 = Float32.(A)
B_32 = Float32.(B)

elapsed = 0
for i in 1:1000 
    start_time = time()
    A * B
    elapsed = elapsed + (time() - start_time)
end

println(elapsed/1000)

elapsed = 0
for i in 1:1000 
    start_time = time()
    A_32 * B_32
    elapsed = elapsed + (time() - start_time)
end

println(elapsed/1000)

elapsed = 0
for i in 1:1000 
    start_time = time()
    A_16 * B_16
    elapsed = elapsed + (time() - start_time)
end

println(elapsed/1000)

0.0008334898948669433
5.40926456451416e-5
8.363175392150879e-5
